# Task 3: Generate Report

Aggregates metrics (total trips, revenue, avg fare/distance), adds the averages from the transform task (`taskValues`) and prints a summary report.

> Same code as in `notebooks/modules/05_orchestration_jobs.ipynb`. `dbutils.notebook.exit()` stays in the last cell on purpose — anything printed in that cell would be hidden in the job run output.


In [0]:
# TASK 3: Generate Report

from pyspark.sql.functions import *
import json
from datetime import datetime

# Parameters
dbutils.widgets.text("source_table", "samples.nyctaxi.trips")

source_table = dbutils.widgets.get("source_table")

# Averages computed by the upstream "transform" task (debugValue is used outside a job)
avg_trip_minutes = dbutils.jobs.taskValues.get(taskKey="transform", key="avg_trip_minutes", debugValue=-1)
avg_cost_per_mile = dbutils.jobs.taskValues.get(taskKey="transform", key="avg_cost_per_mile", debugValue=-1)

# Aggregations
df = spark.table(source_table)

report = df.agg(
    count("*").alias("total_trips"),
    round(sum("fare_amount"), 2).alias("total_revenue"),
    round(avg("fare_amount"), 2).alias("avg_fare"),
    round(avg("trip_distance"), 2).alias("avg_distance"),
    round(max("fare_amount"), 2).alias("max_fare")
).collect()[0]

# Display report
print("\n" + "="*50)
print("DAILY REPORT")
print("="*50)
print(f"Total Trips:    {report.total_trips:,}")
print(f"Total Revenue:  ${report.total_revenue:,.2f}")
print(f"Avg Fare:       ${report.avg_fare:.2f}")
print(f"Avg Distance:   {report.avg_distance:.2f} miles")
print(f"Max Fare:       ${report.max_fare:.2f}")
print(f"Avg Trip:       {avg_trip_minutes:.2f} min   (from transform)")
print(f"Cost per Mile:  ${avg_cost_per_mile:.2f}       (from transform)")
print("="*50)
print(f"Generated at:   {datetime.now()}")
print("="*50 + "\n")


In [0]:
# Return result
dbutils.notebook.exit(json.dumps({
    "status": "SUCCESS",
    "total_trips": report.total_trips,
    "total_revenue": float(report.total_revenue)
}))